In [ ]:
import pynetlogo
import pandas as pd
import itertools
from pathlib import Path

# 1. Explicitly point to the jvm.dll file inside NetLogo's directory
jvm_path = "C:/Program Files/NetLogo 6.4.0/runtime/bin/server/jvm.dll"

netlogo = pynetlogo.NetLogoLink(
    gui=False,
    netlogo_home="C:/Program Files/NetLogo 6.4.0",
    jvm_path=jvm_path 
)

model_path = "C:/Users/15177459/Desktop/netlogo/models/survey_derived.nlogo"

netlogo.load_model(model_path)
netlogo.command("setup")


FileNotFoundError: [Errno 2] JVM DLL not found: /Applications/NetLogo 6.4.0/runtime/lib/server/libjvm.dylib

# Experiment 2: Social Feedback On vs Off

__**NOTES**__: Make a stacked bar chart to see on and off difference

In [20]:
parameter_grid = {
    "number-of-people": [200],
    "baseline-stop-probability": [30],
    "third-place-search-radius": [2],
    "route-third-place-sample-size": [50],
    "rigid-dwell-time": [30],
    "medium-dwell-time": [45],
    "flexible-dwell-time": [60],
    "encounter-odds-increment": [0.10],
    "encounter-weight": [5],
    "social-feedback?": [False, True],
}

reporters = [
    "visits-per-person",
    "average-stop-probability",
    "average-social-encounters",
    "average-social-odds-ratio",
    "total-third-place-visits",
    "total-co-presence",
    "co-presence-per-visit",
    "places-with-co-presence",
    "share-places-visited",
    "rigid-visits-per-person",
    "medium-visits-per-person",
    "flexible-visits-per-person"
]

results = []

keys = list(parameter_grid.keys())
values = list(parameter_grid.values())

run_id = 0

for combination in itertools.product(*values):
    params = dict(zip(keys, combination))
    
    for seed in range(1, 21):  # 20 repetitions
        run_id += 1
        
        netlogo.command(f"random-seed {seed}")
        
        for parameter, value in params.items():
            if isinstance(value, bool):
                value = "true" if value else "false"
            netlogo.command(f"set {parameter} {value}")
        
        netlogo.command("setup")
        netlogo.command("repeat 1000 [ go ]")
        
        row = {
            "experiment": "social_feedback_test",
            "run_id": run_id,
            "seed": seed,
            **params,
        }
        
        for reporter in reporters:
            row[reporter] = netlogo.report(reporter)
        
        results.append(row)

exp2 = pd.DataFrame(results)

output_path = "C:/Users/15177459/Desktop/netlogo/models/outputs/experiment_2_social_feedback.csv"
exp2.to_csv(output_path, index=False)

print(exp2.shape)
exp2.head()

(40, 25)


,experiment,run_id,seed,number-of-people,baseline-stop-probability,third-place-search-radius,route-third-place-sample-size,rigid-dwell-time,medium-dwell-time,flexible-dwell-time,...,average-social-encounters,average-social-odds-ratio,total-third-place-visits,total-co-presence,co-presence-per-visit,places-with-co-presence,share-places-visited,rigid-visits-per-person,medium-visits-per-person,flexible-visits-per-person
0,social_feedback_test,1,1,200,30,2,50,30,45,60,...,4.50,1.0,339.0,3530.0,10.412979,41.0,0.983333,1.410959,1.850000,1.865672
1,social_feedback_test,2,2,200,30,2,50,30,45,60,...,4.20,1.0,300.0,3190.0,10.633333,41.0,0.983333,1.159420,1.514706,1.857143
2,social_feedback_test,3,3,200,30,2,50,30,45,60,...,6.20,1.0,370.0,4849.0,13.105405,37.0,0.983333,1.776316,1.833333,1.953125
3,social_feedback_test,4,4,200,30,2,50,30,45,60,...,4.35,1.0,325.0,3091.0,9.510769,38.0,1.000000,1.357143,1.750000,1.787879
4,social_feedback_test,5,5,200,30,2,50,30,45,60,...,3.70,1.0,333.0,3053.0,9.168168,29.0,1.000000,1.734375,1.515625,1.736111


In [21]:
exp2_summary = (
    exp2.groupby("social-feedback?")
    [[
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-third-place-visits",
        "total-co-presence",
        "co-presence-per-visit",
        "places-with-co-presence",
        "share-places-visited",
        "rigid-visits-per-person",
        "medium-visits-per-person",
        "flexible-visits-per-person"
    ]]
    .agg(["mean", "std"])
)

exp2_summary

visits-per-person           average-stop-probability  \
                              mean       std                     mean   
social-feedback?                                                        
False                      1.72500  0.114467                22.876040   
True                       1.85075  0.126723                29.801917   

                           average-social-encounters            \
                       std                      mean       std   
social-feedback?                                                 
False             0.380643                    4.9500  0.752540   
True              1.078257                    5.4925  0.901793   

                 average-social-odds-ratio           total-third-place-visits  \
                                      mean       std                     mean   
social-feedback?                                                                
False                              1.00000  0.000000                   345.00   
True                               1.49825  0.070406                   370.15   

                             ... places-with-co-presence            \
                        std  ...                    mean       std   
social-feedback?             ...                                     
False             22.893345  ...                   40.05  3.748333   
True              25.344521  ...                   40.95  3.119970   

                 share-places-visited           rigid-visits-per-person  \
                                 mean       std                    mean   
social-feedback?                                                          
False                        0.993333  0.011343                1.549725   
True                         0.994167  0.011180                1.661323   

                           medium-visits-per-person            \
                       std                     mean       std   
social-feedback?                                                
False             0.261116                 1.759253  0.186139   
True              0.280037                 1.873381  0.203989   

                 flexible-visits-per-person            
                                       mean       std  
social-feedback?                                       
False                              1.877451  0.182838  
True                               2.027690  0.221704  

[2 rows x 24 columns]

In [22]:
exp2_table = (
    exp2.groupby("social-feedback?")
    [[
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit",
        "places-with-co-presence"
    ]]
    .mean()
    .reset_index()
)

exp2_table

,social-feedback?,visits-per-person,average-stop-probability,average-social-encounters,average-social-odds-ratio,total-co-presence,co-presence-per-visit,places-with-co-presence
0,False,1.72500,22.876040,4.9500,1.00000,3844.15,11.095843,40.05
1,True,1.85075,29.801917,5.4925,1.49825,4276.70,11.525644,40.95


In [23]:
off = exp2_table[exp2_table["social-feedback?"] == False].iloc[0]
on = exp2_table[exp2_table["social-feedback?"] == True].iloc[0]

comparison = pd.DataFrame({
    "metric": [
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit",
        "places-with-co-presence"
    ],
    "feedback_off": [
        off["visits-per-person"],
        off["average-stop-probability"],
        off["average-social-encounters"],
        off["average-social-odds-ratio"],
        off["total-co-presence"],
        off["co-presence-per-visit"],
        off["places-with-co-presence"]
    ],
    "feedback_on": [
        on["visits-per-person"],
        on["average-stop-probability"],
        on["average-social-encounters"],
        on["average-social-odds-ratio"],
        on["total-co-presence"],
        on["co-presence-per-visit"],
        on["places-with-co-presence"]
    ],
})

comparison["absolute_change"] = comparison["feedback_on"] - comparison["feedback_off"]
comparison["percent_change"] = (
    comparison["absolute_change"] / comparison["feedback_off"] * 100
)

comparison

,metric,feedback_off,feedback_on,absolute_change,percent_change
0,visits-per-person,1.725000,1.850750,0.125750,7.289855
1,average-stop-probability,22.876040,29.801917,6.925878,30.275686
2,average-social-encounters,4.950000,5.492500,0.542500,10.959596
3,average-social-odds-ratio,1.000000,1.498250,0.498250,49.825000
4,total-co-presence,3844.150000,4276.700000,432.550000,11.252162
5,co-presence-per-visit,11.095843,11.525644,0.429801,3.873531
6,places-with-co-presence,40.050000,40.950000,0.900000,2.247191


# Experiment 3 = mobility / route-based accessibility test.

_**NOTES**_: Choose one of the reporters and make a scatter plot. Reporter in y-axis and third-place-search-radius in x-axis

In [16]:
# Experiment 3: mobility / route-based accessibility scenario
parameter_grid = {
    "number-of-people": [200],
    "baseline-stop-probability": [30],
    "third-place-search-radius": [1, 2, 5, 10],
    "route-third-place-sample-size": [50],
    "rigid-dwell-time": [30],
    "medium-dwell-time": [45],
    "flexible-dwell-time": [60],
    "encounter-odds-increment": [0.10],
    "encounter-weight": [5],
    "social-feedback?": [True],
}

reporters = [
    "average-route-third-place-options",
    "people-with-route-third-place-options",
    "visits-per-person",
    "average-stop-probability",
    "average-social-encounters",
    "average-social-odds-ratio",
    "total-third-place-visits",
    "total-co-presence",
    "co-presence-per-visit",
    "places-with-co-presence",
    "share-places-visited",
    "rigid-visits-per-person",
    "medium-visits-per-person",
    "flexible-visits-per-person"
]

results = []

keys = list(parameter_grid.keys())
values = list(parameter_grid.values())

run_id = 0

for combination in itertools.product(*values):
    params = dict(zip(keys, combination))
    
    for seed in range(1, 21):  # 20 repetitions
        run_id += 1
        
        netlogo.command(f"random-seed {seed}")
        
        for parameter, value in params.items():
            if isinstance(value, bool):
                value = "true" if value else "false"
            netlogo.command(f"set {parameter} {value}")
        
        netlogo.command("setup")
        netlogo.command("repeat 1000 [ go ]")
        
        row = {
            "experiment": "mobility_accessibility_test",
            "run_id": run_id,
            "seed": seed,
            **params,
        }
        
        for reporter in reporters:
            row[reporter] = netlogo.report(reporter)
        
        results.append(row)

exp3 = pd.DataFrame(results)

output_path = "C:/Users/15177459/Desktop/netlogo/models/outputs/experiment_3_mobility_accessibility.csv"
exp3.to_csv(output_path, index=False)

print(exp3.shape)
exp3.head()

(80, 27)


,experiment,run_id,seed,number-of-people,baseline-stop-probability,third-place-search-radius,route-third-place-sample-size,rigid-dwell-time,medium-dwell-time,flexible-dwell-time,...,average-social-encounters,average-social-odds-ratio,total-third-place-visits,total-co-presence,co-presence-per-visit,places-with-co-presence,share-places-visited,rigid-visits-per-person,medium-visits-per-person,flexible-visits-per-person
0,mobility_accessibility_test,1,1,200,30,1,50,30,45,60,...,5.80,1.5250,397.0,4184.0,10.539043,47.0,1.000000,1.821918,2.033333,2.119403
1,mobility_accessibility_test,2,2,200,30,1,50,30,45,60,...,5.30,1.4240,342.0,3698.0,10.812865,31.0,0.966667,1.478261,1.617647,2.063492
2,mobility_accessibility_test,3,3,200,30,1,50,30,45,60,...,10.10,1.6805,487.0,6376.0,13.092402,41.0,1.000000,2.394737,2.533333,2.390625
3,mobility_accessibility_test,4,4,200,30,1,50,30,45,60,...,5.00,1.4185,349.0,3683.0,10.553009,31.0,0.950000,1.285714,2.093750,1.893939
4,mobility_accessibility_test,5,5,200,30,1,50,30,45,60,...,7.75,1.5885,413.0,5962.0,14.435835,46.0,1.000000,2.046875,2.125000,2.027778


In [17]:
# ------------------------------------------------------------
# EXPERIMENT 3 ANALYSIS
# ------------------------------------------------------------


# Group results by search radius and calculate mean + standard deviation.
# This tells us the average model outcome for each mobility scenario.

exp3_summary = (
    exp3.groupby("third-place-search-radius")
    [[
        "average-route-third-place-options",
        "people-with-route-third-place-options",
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-third-place-visits",
        "total-co-presence",
        "co-presence-per-visit",
        "places-with-co-presence",
        "share-places-visited"
    ]]
    .agg(["mean", "std"])
)

exp3_summary

average-route-third-place-options            \
                                                       mean       std   
third-place-search-radius                                               
1                                                  16.44125  0.954167   
2                                                  35.22925  1.095994   
5                                                  58.06325  0.281206   
10                                                 60.00000  0.000000   

                          people-with-route-third-place-options            \
                                                           mean       std   
third-place-search-radius                                                   
1                                                        199.95  0.223607   
2                                                        200.00  0.000000   
5                                                        200.00  0.000000   
10                                                       200.00  0.000000   

                          visits-per-person            \
                                       mean       std   
third-place-search-radius                               
1                                   2.08525  0.184366   
2                                   1.85075  0.126723   
5                                   1.67625  0.133858   
10                                  1.61425  0.069116   

                          average-stop-probability            \
                                              mean       std   
third-place-search-radius                                      
1                                        30.856953  1.399253   
2                                        29.801917  1.078257   
5                                        28.934160  1.221778   
10                                       28.423165  0.858840   

                          average-social-encounters            ...  \
                                               mean       std  ...   
third-place-search-radius                                      ...   
1                                            7.8050  1.870400  ...   
2                                            5.4925  0.901793  ...   
5                                            4.5525  0.927004  ...   
10                                           4.0650  0.603302  ...   

                          total-third-place-visits             \
                                              mean        std   
third-place-search-radius                                       
1                                           417.05  36.873147   
2                                           370.15  25.344521   
5                                           335.25  26.771696   
10                                          322.85  13.823226   

                          total-co-presence               \
                                       mean          std   
third-place-search-radius                                  
1                                    5590.4  1245.282356   
2                                    4276.7   668.780595   
5                                    3586.2   734.167384   
10                                   3241.9   462.601670   

                          co-presence-per-visit            \
                                           mean       std   
third-place-search-radius                                   
1                                     13.283782  2.060962   
2                                     11.525644  1.401103   
5                                     10.617254  1.633484   
10                                    10.027009  1.239500   

                          places-with-co-presence            \
                                             mean       std   
third-place-search-radius                                     
1                                           40.40  4.627151   
2                                           40.95  3.119970   
5             

In [19]:
# Create a cleaner table with means only.
# This is easier to paste into a thesis table.

exp3_table = (
    exp3.groupby("third-place-search-radius")
    [[
        "average-route-third-place-options",
        "people-with-route-third-place-options",
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit",
        "places-with-co-presence"
    ]]
    .mean()
    .reset_index()
)

# Add readable scenario labels.
radius_labels = {
    1: "Very low route-based accessibility",
    2: "Low route-based accessibility",
    5: "Medium route-based accessibility",
    10: "High route-based accessibility"
}

exp3_table["scenario"] = exp3_table["third-place-search-radius"].map(radius_labels)

# Reorder columns so the scenario label appears first.
exp3_table = exp3_table[
    [
        "scenario",
        "third-place-search-radius",
        "average-route-third-place-options",
        "people-with-route-third-place-options",
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit",
        "places-with-co-presence"
    ]
]

exp3_table

,scenario,third-place-search-radius,average-route-third-place-options,people-with-route-third-place-options,visits-per-person,average-stop-probability,average-social-encounters,average-social-odds-ratio,total-co-presence,co-presence-per-visit,places-with-co-presence
0,Very low route-based accessibility,1,16.44125,199.95,2.08525,30.856953,7.8050,1.586500,5590.4,13.283782,40.40
1,Low route-based accessibility,2,35.22925,200.00,1.85075,29.801917,5.4925,1.498250,4276.7,11.525644,40.95
2,Medium route-based accessibility,5,58.06325,200.00,1.67625,28.934160,4.5525,1.427975,3586.2,10.617254,38.85
3,High route-based accessibility,10,60.00000,200.00,1.61425,28.423165,4.0650,1.388450,3241.9,10.027009,36.95
